# Running the dashboard locally

## Setup

Refer to [README.md](README.md) to correctly create your Python virtual environment. 

After creating and activating the Python venv, running the next code cell should work without issue.

In [1]:
%pip install -q click==8.4.1 pyyaml requests==2.34.2 hugo==0.162.0.1

import os
import importlib
from pathlib import Path

from data_processing import load_existing_journals
from utils import ensure_dir

importlib.invalidate_caches()
print('✅ Installed dependencies.')


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ Installed dependencies.


Most of the following cells will use the **Data Journals Dashboard** CLI helper `dj` to fetch and process data journals metadata and build the Hugo websites. 

In order to prevent overwriting any existing files, we will define a output directory in the `tests` folder of the repository's working dir:

In [2]:
PROJECT_ROOT = Path(os.getcwd()).parent
SCHEMA_PATH = PROJECT_ROOT / "metadata_schema" / "schema.yaml"
OUTPUT_DIR = PROJECT_ROOT / "tests" / "output"

ensure_dir(OUTPUT_DIR)  # Create OUTPUT_DIR if it does not exist

Test if the CLI helper is correctly installed in our environment.

In [3]:
!dj 

Usage: dj [OPTIONS] COMMAND [ARGS]...

  Data Journal Dashboard CLI Helper.

Options:
  --help  Show this message and exit.

Commands:
  collect   Fetch or parse raw journal metadata from GitHub or a local...
  process   Process raw journal metadata by validating it against the...
  hugo      Generate Hugo static site content from processed journal data.
  export    Export data journal metadata to different file types.
  validate  Validate journals against metadata schema.


## Collecting journal metadata from GitHub

The dashboard’s primary data source are data journal metadata published under CC0 1.0 Universal on Zenodo and Github.

>Kindling, Maxi, and Dorothea Strecker. 2022. List of Data Journals (1.0). [https://doi.org/10.5281/zenodo.7082126](https://doi.org/10.5281/zenodo.7082126). 

To collect the `csv` data from GitHub we use the CLI helper's `collect` command and pass our `OUTPUT_DIR` path:

In [5]:
!dj collect --output_fpath $OUTPUT_DIR

Fetching data journal data from GitHub...
Saved raw CSV → /Users/thomas/Coding/sandbox/data-journals-dashboard/tests/output/data_journals.csv


We can inspect the downloaded `csv`:

In [6]:
!head -n 5 ../tests/output/data_journals.csv

"ISSN","journal_title","publisher","URL","data_journal_type"
"2574-5417","Big Earth Data","Taylor & Francis","https://www.tandfonline.com/journals/tbed20","mixed"
"1809-127X","Check List","Pensoft","https://checklist.pensoft.net/","pure"
"1698-0476","Arxius de Miscel·lània Zoològica","Museu de Ciències Naturals de Barcelona","https://museucienciesjournals.cat/en/amz","pure"
"1941-4927","NCHS Data Briefs","Centers for Disease Control and Preventions (CDC)","https://www.cdc.gov/nchs/products/databriefs.htm","pure"


## Metadata processing

We process the raw `csv` metadata from GitHub by adding additional metadata from the [Directory of Open Access Journal's public API](https://doaj.org/api/v4/docs) with the CLI helper's `process all` command. 

For each journal in the `csv` the command will send a GET request using the Python `requests` package to the DOAJs API endpoint "https://doaj.org/api/search/journals/" and add the journals ISSN. If the ISSN is found, all DOAJ metadata are appended to the existing `csv` metadata.

The primary processing function is `enrich_journals_with_doaj()` in `../src/data_processing.py`.

In [7]:
!dj process all --help

Usage: dj process all [OPTIONS]

  Process all journals in one go.

Options:
  -i, --input_fpath PATH   Path to the journal metadata CSV file.  [default:
                           data/raw/data_journals.csv]
  -s, --schema_path PATH   Path to the journal metadata schema YAML file.
                           [default: metadata_schema/schema.yaml]
  -o, --output_fpath PATH  Path to save the processed journal collection YAML
                           file.  [default: data/data_journals.yaml]
  --help                   Show this message and exit.


In [8]:
!dj process all \
    --input_fpath ../tests/output/data_journals.csv \
    --output_fpath $OUTPUT_DIR \
    --schema_path $SCHEMA_PATH

Parsed 142 journals.
[1/142] Adding metadata from doaj.org to 2574-5417...
[2/142] Adding metadata from doaj.org to 1809-127X...
[3/142] Adding metadata from doaj.org to 1698-0476...
[4/142] Adding metadata from doaj.org to 1941-4927...
  No DOAJ entry found for 1941-4927.
[5/142] Adding metadata from doaj.org to 1758-0463...
[6/142] Adding metadata from doaj.org to 2709-4715...
[7/142] Adding metadata from doaj.org to 2572-4754...
[8/142] Adding metadata from doaj.org to 2732-5121...
[9/142] Adding metadata from doaj.org to 2363-4952...
[10/142] Adding metadata from doaj.org to 2059-481X...
[11/142] Adding metadata from doaj.org to 2296-7745...
[12/142] Adding metadata from doaj.org to 2054-345X...
[13/142] Adding metadata from doaj.org to 2032-6378...
  No DOAJ entry found for 2032-6378.
[14/142] Adding metadata from doaj.org to 1297-966X...
[15/142] Adding metadata from doaj.org to 0092-640X...
  No DOAJ entry found for 0092-640X.
[16/142] Adding metadata from doaj.org to 1314-2828.

The processed journal metadata is saved as a `.yaml` file that we can load and inspect:

In [9]:
journal_collection = load_existing_journals(fpath=OUTPUT_DIR / "data_journals.yaml")

# Inspect the first journal of the collection with complete metadata
journal_collection[0]

{'id': 1,
 'is_active': True,
 'enrichment_source': 'doaj',
 'issn': '2574-5417',
 'journal_title': 'Big Earth Data',
 'publisher': 'Taylor & Francis',
 'url': 'https://www.tandfonline.com/journals/tbed20',
 'data_journal_type': 'mixed',
 'eissn': '2574-5417',
 'pissn': '2096-4471',
 'language': ['EN'],
 'oa_start': 2017,
 'boai': True,
 'publication_time_weeks': 10,
 'keywords': ['geography',
  'geology',
  'atmospheric science',
  'marine science',
  'earth system science',
  'big data studies'],
 'research_fields': ['Geography. Anthropology. Recreation', 'Geology'],
 'subject_codes': [{'code': 'G',
   'scheme': 'LCC',
   'term': 'Geography. Anthropology. Recreation'},
  {'code': 'QE1-996.5', 'scheme': 'LCC', 'term': 'Geology'}],
 'publisher_name': 'Taylor & Francis Group',
 'publisher_country': 'GB',
 'institution_name': 'The International Society for Digital Earth',
 'institution_country': 'CN',
 'review_process': ['Anonymous peer review'],
 'review_url': 'https://www.tandfonline.c

## Building the website with Hugo

### Creating markdown documents for each journal

Hugo will generate `html` webpages from markdown documents in `../content/journals` and theme templates located in `../themes`. We first need to create the markdown documents. The processing input is our `dadata_journals.yaml` collection. The CLI helper's `generate` command automatically creates the markdown documents by parsing the yaml metadata and adding the metadata as Hugo front matter to the documents.

The parsing and generation logic can be inspected in the `create_journal_content_for_hugo()` function in `../src/hugo_transform.py`.

In [11]:
!dj hugo generate --help

Usage: dj hugo generate [OPTIONS]

  Generate Hugo-compatible markdown files from processed journal data.

Options:
  -i, --input_fpath PATH  Path to the processed journal YAML file.  [default:
                          data/data_journals.yaml]
  -o, --output_dir PATH   Hugo site root directory.  [default: .]
  -s, --schema_path PATH  Path to the journal metadata schema YAML file.
                          [default: metadata_schema/schema.yaml]
  --help                  Show this message and exit.


In [12]:
!dj hugo generate \
    --input_fpath ../tests/output/data_journals.yaml \
    --output_dir ../ \
    --schema_path $SCHEMA_PATH

Generated: 25745417-big-earth-data.md
Generated: 1809127X-check-list.md
Generated: 16980476-arxius-de-miscellània-zoològica.md
Generated: 19414927-nchs-data-briefs.md
Generated: 17580463-database-the-journal-of-biological-databases-and-c.md
Generated: 27094715-gigabyte.md
Generated: 25724754-gates-open-research.md
Generated: 27325121-open-research-europe.md
Generated: 23634952-ride---a-review-journal-for-digital-editions-and-r.md
Generated: 2059481X-journal-of-open-humanities-data.md
Generated: 22967745-frontiers-in-marine-science.md
Generated: 2054345X-human-genome-variation.md
Generated: 20326378-journal-of-astronomical-data.md
Generated: 1297966X-annals-of-forest-science.md
Generated: 0092640X-atomic-data-and-nuclear-data-tables.md
Generated: 13142828-biodiversity-data-journal.md
Generated: 22421300-bioinvasions-records.md
Generated: 20426410-biology-of-sex-differences.md
Generated: 14712253-bmc-anesthesiology.md
Generated: 14712105-bmc-bioinformatics.md
Generated: 17417007-bmc-biol

Inspect one of the created markdown documents:

In [13]:
!cat ../content/journals/25745417-big-earth-data.md

---
title: Big Earth Data
date: '2026-06-02'
draft: false
issn: 2574-5417
is_active: active
data_journal_type:
- mixed
publisher:
- Taylor & Francis
keywords:
- geography
- geology
- atmospheric science
- marine science
- earth system science
- big data studies
research_fields:
- Geography. Anthropology. Recreation
- Geology
license_types:
- CC BY
- CC BY-NC
apc_has: 'yes'
apc_price_range: 1200 EUR, 1250 USD, 1000 GBP
preservation_services:
- CLOCKSS
- LOCKSS
- Portico
boai: 'yes'
id: 1
enrichment_source: doaj
journal_title: Big Earth Data
journal_url: https://www.tandfonline.com/journals/tbed20
eissn: 2574-5417
pissn: 2096-4471
language:
- EN
oa_start: 2017
publication_time_weeks: 10
subject_codes:
- code: G
  scheme: LCC
  term: Geography. Anthropology. Recreation
- code: QE1-996.5
  scheme: LCC
  term: Geology
publisher_name: Taylor & Francis Group
publisher_country: GB
institution_name: The International Society for Digital Earth
institution_country: CN
review_process:
- Anonymous 

As described above, the markdown journal page contains a `yaml` header which Hugo will parse as front matter for each page and some additional basic metadata in the markdown body.

### Generating the Hugo websites

With `hugo build` we now create the final `html` websites:

In [17]:
!hugo build --source $PROJECT_ROOT

Running Hugo 0.162.0.1 via hugo-python-distributions at /Users/thomas/Coding/sandbox/data-journals-dashboard/venv_py313/hugo/binaries/hugo
]9;4;0;0Start building sites … 
hugo v0.162.0-076dfe13d0f789e3d9586b192f8f7f3329c26990+extended+withdeploy darwin/arm64 BuildDate=2026-05-26T13:53:44Z VendorInfo=hugo-python-distributions

;4;1;15]9;4;1;20]9;4;1;100]9;4;3;100]9;4;1;100]9;4;0;100
                  │  EN  
──────────────────┼──────
 Pages            │ 1062 
 Paginator pages  │    0 
 Non-page files   │    0 
 Static files     │    2 
 Processed images │    0 
 Aliases          │    0 
 Cleaned          │    0 

Total in 290 ms


`hugo server` will start a local web server that runs our website. The web server can be stopped by stopping the notebooks cell.

In [19]:
!hugo server --source $PROJECT_ROOT

Running Hugo 0.162.0.1 via hugo-python-distributions at /Users/thomas/Coding/sandbox/data-journals-dashboard/venv_py313/hugo/binaries/hugo
port 1313 already in use, attempting to use an available port
]9;4;0;0Watching for changes in /Users/thomas/Coding/sandbox/data-journals-dashboard/assets, /Users/thomas/Coding/sandbox/data-journals-dashboard/content/journals, /Users/thomas/Coding/sandbox/data-journals-dashboard/hugo-data, /Users/thomas/Coding/sandbox/data-journals-dashboard/themes/djd/archetypes, /Users/thomas/Coding/sandbox/data-journals-dashboard/themes/djd/layouts/{_default,page,partials,taxonomy}, /Users/thomas/Coding/sandbox/data-journals-dashboard/themes/djd/static/{css,logos}
Watching for config changes in /Users/thomas/Coding/sandbox/data-journals-dashboard/hugo.toml
Start building sites … 
hugo v0.162.0-076dfe13d0f789e3d9586b192f8f7f3329c26990+extended+withdeploy darwin/arm64 BuildDate=2026-05-26T13:53:44Z VendorInfo=hugo-python-distributions

25h
                  │  EN 

OSError: [Errno 5] Input/output error

## Notes

This notebook demonstrates both pipelines of the **Data Journals Dashboard**: 

1. Data collection and processing
2. Website creation with Hugo

As described above, the metadata collection `data_journals.yaml` — which has been processed using data from the **Directory of Open Access Journals** — serves as the basis for website creation. For the production instance of the Data Journals Dashboard, this collection was semi-automatically enhanced (i.e., additional metadata was added based on the journal homepage, Wikidata, etc.). This is why certain page contents of the production instance differ from the data journal collection shown in this notebook.